In [1]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Week 4: Statistical Analysis and Hypothesis Testing

This analysis applies statistical methods to the Titanic dataset to test relationships between passenger characteristics and survival outcomes.

The analysis focuses on gender, passenger class, age, and ticket fare.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math

In [3]:
df = pd.read_csv("C:/Users/PRAKRUTHI/OneDrive/Desktop/DA_Internship/Titanic_Cleaned.csv")

print("Shape:", df.shape)
print(df.head())

Shape: (418, 11)
   PassengerId  Survived  Pclass  \
0          892         0       3   
1          893         1       3   
2          894         0       2   
3          895         0       3   
4          896         1       3   

                                           Name     Sex   Age  SibSp  Parch  \
0                              Kelly, Mr. James    male  34.5      0      0   
1              Wilkes, Mrs. James (Ellen Needs)  female  47.0      1      0   
2                     Myles, Mr. Thomas Francis    male  62.0      0      0   
3                              Wirz, Mr. Albert    male  27.0      0      0   
4  Hirvonen, Mrs. Alexander (Helga E Lindqvist)  female  22.0      1      1   

    Ticket     Fare Embarked  
0   330911   7.8292        Q  
1   363272   7.0000        S  
2   240276   9.6875        Q  
3   315154   8.6625        S  
4  3101298  12.2875        S  


## 1. Data Validation

The dataset is checked before performing statistical analysis. The shape, columns, and missing values are reviewed to ensure the data is suitable for analysis.

In [4]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          418 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         418 non-null    float64
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(4)
memory usage: 36.1+ KB
None

Missing Values:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


## 2. Significance Level

A significance level of 0.05 is used for all hypothesis tests.

If the p-value is less than 0.05, the null hypothesis will be rejected.

If the p-value is greater than or equal to 0.05, the null hypothesis will not be rejected.

In [5]:
alpha = 0.05

print("Significance level:", alpha)

Significance level: 0.05


## 3. Hypothesis 1: Gender and Survival

Research Question:

Is passenger gender associated with survival?

H0: Gender and survival are independent.

H1: Gender and survival are associated.

Statistical Test:

Chi-square test of independence.

The chi-square test is suitable because both Gender and Survival are categorical variables.

In [6]:
gender_table = pd.crosstab(df["Sex"], df["Survived"])

print("Gender vs Survival:")
print(gender_table)

chi2_gender, p_gender, dof_gender, expected_gender = stats.chi2_contingency(
    gender_table
)

n = gender_table.values.sum()

cramers_v_gender = np.sqrt(
    chi2_gender /
    (n * min(gender_table.shape[0] - 1, gender_table.shape[1] - 1))
)

print("\nChi-square:", chi2_gender)
print("Degrees of freedom:", dof_gender)
print("p-value:", p_gender)
print("Cramer's V:", cramers_v_gender)

if p_gender < alpha:
    print("Decision: Reject H0")
else:
    print("Decision: Do not reject H0")

Gender vs Survival:
Survived    0    1
Sex               
female      0  152
male      266    0

Chi-square: 413.6897405343716
Degrees of freedom: 1
p-value: 5.767311139789629e-92
Cramer's V: 0.9948308270676691
Decision: Reject H0


### Interpretation

The chi-square test produced a p-value of approximately 5.77 × 10⁻⁹², which is far below the significance level of 0.05.

Therefore, the null hypothesis is rejected.

There is a statistically significant association between gender and survival in this dataset.

Cramer's V is approximately 0.995, indicating a very strong association between gender and survival.

## 4. Hypothesis 2: Passenger Class and Survival

Research Question:

Is passenger class associated with survival?

H0: Passenger class and survival are independent.

H1: Passenger class and survival are associated.

Statistical Test:

Chi-square test of independence.

In [7]:
pclass_table = pd.crosstab(df["Pclass"], df["Survived"])

print("Passenger Class vs Survival:")
print(pclass_table)

chi2_pclass, p_pclass, dof_pclass, expected_pclass = stats.chi2_contingency(
    pclass_table
)

n = pclass_table.values.sum()

cramers_v_pclass = np.sqrt(
    chi2_pclass /
    (n * min(pclass_table.shape[0] - 1, pclass_table.shape[1] - 1))
)

print("\nChi-square:", chi2_pclass)
print("Degrees of freedom:", dof_pclass)
print("p-value:", p_pclass)
print("Cramer's V:", cramers_v_pclass)

if p_pclass < alpha:
    print("Decision: Reject H0")
else:
    print("Decision: Do not reject H0")

Passenger Class vs Survival:
Survived    0   1
Pclass           
1          57  50
2          63  30
3         146  72

Chi-square: 6.693869422819262
Degrees of freedom: 2
p-value: 0.03519206276590605
Cramer's V: 0.12654659885348873
Decision: Reject H0


### Interpretation

The chi-square test produced a p-value of 0.0352, which is below the significance level of 0.05.

Therefore, the null hypothesis is rejected.

There is a statistically significant association between passenger class and survival in this dataset.

However, Cramer's V is approximately 0.127, indicating that the strength of this association is relatively weak.

## 5. Hypothesis 3: Age and Survival

Research Question:

Does passenger age differ between survivors and non-survivors?

H0: The mean age is the same for survivors and non-survivors.

H1: The mean age differs between survivors and non-survivors.

Statistical Test:

Welch's independent samples t-test.

In [8]:
age_survived = df.loc[df["Survived"] == 1, "Age"].dropna()
age_not_survived = df.loc[df["Survived"] == 0, "Age"].dropna()

print("Survivor mean age:", age_survived.mean())
print("Non-survivor mean age:", age_not_survived.mean())

t_age, p_age = stats.ttest_ind(
    age_not_survived,
    age_survived,
    equal_var=False
)

print("\nWelch t-statistic:", t_age)
print("p-value:", p_age)

if p_age < alpha:
    print("Decision: Reject H0")
else:
    print("Decision: Do not reject H0")

Survivor mean age: 30.27239973050095
Non-survivor mean age: 30.272699293414263

Welch t-statistic: 0.00022170853965751481
p-value: 0.9998232656457627
Decision: Do not reject H0


### Interpretation

The Welch t-test produced a p-value of approximately 0.9998, which is greater than the significance level of 0.05.

Therefore, the null hypothesis is not rejected.

There is no statistically significant difference in mean age between survivors and non-survivors in this dataset.

The mean ages of the two groups are almost identical.

## 6. Hypothesis 4: Fare and Survival

Research Question:

Does ticket fare differ between survivors and non-survivors?

H0: The mean fare is the same for survivors and non-survivors.

H1: The mean fare differs between survivors and non-survivors.

Statistical Test:

Welch's independent samples t-test.

In [10]:
fare_survived = df.loc[df["Survived"] == 1, "Fare"].dropna()
fare_not_survived = df.loc[df["Survived"] == 0, "Fare"].dropna()

print("Survivor mean fare:", fare_survived.mean())
print("Non-survivor mean fare:", fare_not_survived.mean())

t_fare, p_fare = stats.ttest_ind(
    fare_not_survived,
    fare_survived,
    equal_var=False
)

print("\nWelch t-statistic:", t_fare)
print("p-value:", p_fare)

if p_fare < alpha:
    print("Decision: Reject H0")
else:
    print("Decision: Do not reject H0")

Survivor mean fare: 49.747698684210526
Non-survivor mean fare: 27.558325520636124

Welch t-statistic: -3.445091501468687
p-value: 0.000691442702432026
Decision: Reject H0


## 6. Hypothesis 5: Embarked Port and Survival

### Research Question

Does passenger survival differ based on the port of embarkation?

### H0: Null Hypothesis

There is no significant association between embarked port and passenger survival.

### H1: Alternative Hypothesis

There is a significant association between embarked port and passenger survival.

### Statistical Test

Chi-square test of independence.

### Significance Level

Alpha = 0.05

In [11]:
embarked_table = pd.crosstab(df["Embarked"], df["Survived"])

print("Embarked Port vs Survival:")
print(embarked_table)

chi2_embarked, p_embarked, dof_embarked, expected_embarked = stats.chi2_contingency(
    embarked_table
)

n = embarked_table.values.sum()

cramers_v_embarked = np.sqrt(
    chi2_embarked /
    (n * min(embarked_table.shape[0] - 1, embarked_table.shape[1] - 1))
)

print("\nChi-square:", chi2_embarked)
print("Degrees of freedom:", dof_embarked)
print("p-value:", p_embarked)
print("Cramer's V:", cramers_v_embarked)

if p_embarked < alpha:
    print("Decision: Reject H0")
else:
    print("Decision: Do not reject H0")

Embarked Port vs Survival:
Survived    0   1
Embarked         
C          62  40
Q          22  24
S         182  88

Chi-square: 6.9867467760050905
Degrees of freedom: 2
p-value: 0.030398154246084726
Cramer's V: 0.12928536346297598
Decision: Reject H0


### Interpretation

The chi-square test evaluates whether passenger survival was associated with the port of embarkation.

If the p-value is less than 0.05, the null hypothesis is rejected, indicating a statistically significant association between embarkation port and survival in the dataset.

If the p-value is greater than or equal to 0.05, the null hypothesis is not rejected, indicating insufficient statistical evidence of an association between embarkation port and survival.

Cramer's V is used to describe the strength of the association.

### Result

The p-value is 0.0304, which is less than the significance level of 0.05.

Therefore, the null hypothesis is rejected.

There is a statistically significant association between the port of embarkation and passenger survival in the dataset.

Cramer's V is 0.1293, indicating a weak association between the two variables.

## 7. Hypothesis 6: Family Size and Survival

### Research Question

Does passenger survival differ based on family size?

### H0: Null Hypothesis

There is no significant association between family size and passenger survival.

### H1: Alternative Hypothesis

There is a significant association between family size and passenger survival.

### Statistical Test

Chi-square test of independence.

### Significance Level

Alpha = 0.05

In [14]:
print(df.columns.tolist())

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']


In [15]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

print(df[["SibSp", "Parch", "FamilySize"]].head())

   SibSp  Parch  FamilySize
0      0      0           1
1      1      0           2
2      0      0           1
3      0      0           1
4      1      1           3


In [17]:
family_table = pd.crosstab(df["FamilySize"], df["Survived"])

print("Family Size vs Survival:")
print(family_table)

chi2_family, p_family, dof_family, expected_family = stats.chi2_contingency(
    family_table
)

n = family_table.values.sum()

cramers_v_family = np.sqrt(
    chi2_family /
    (n * min(family_table.shape[0] - 1, family_table.shape[1] - 1))
)

print("\nChi-square:", chi2_family)
print("Degrees of freedom:", dof_family)
print("p-value:", p_family)
print("Cramer's V:", cramers_v_family)

if p_family < alpha:
    print("Decision: Reject H0")
else:
    print("Decision: Do not reject H0")

Family Size vs Survival:
Survived      0   1
FamilySize         
1           185  68
2            38  36
3            27  30
4             4  10
5             5   2
6             1   2
7             3   1
8             1   1
11            2   2

Chi-square: 30.701843448431383
Degrees of freedom: 8
p-value: 0.0001587440495571054
Cramer's V: 0.27101547232633505
Decision: Reject H0


### Result

The p-value is 0.000159, which is less than the significance level of 0.05.

Therefore, the null hypothesis is rejected.

There is a statistically significant association between family size and passenger survival in the dataset.

Cramer's V is 0.2710, indicating a weak to moderate association between family size and survival.

## 8. Overall Hypothesis Testing Summary

Six statistical hypotheses were tested using the Titanic dataset.

1. Gender and Survival
The chi-square test showed a statistically significant association between gender and survival. The null hypothesis was rejected.

2. Passenger Class and Survival
The chi-square test showed a statistically significant association between passenger class and survival. The null hypothesis was rejected.

3. Age and Survival
Welch's independent samples t-test showed no statistically significant difference in mean age between survivors and non-survivors. The null hypothesis was not rejected.

4. Fare and Survival
Welch's independent samples t-test showed a statistically significant difference in mean fare between survivors and non-survivors. The null hypothesis was rejected.

5. Embarked Port and Survival
The chi-square test showed a statistically significant association between embarkation port and survival. The null hypothesis was rejected.

6. Family Size and Survival
The chi-square test showed a statistically significant association between family size and survival. The null hypothesis was rejected.

Overall, the analysis indicates that gender, passenger class, fare, embarkation port, and family size were statistically associated with survival in this dataset. Age did not show a statistically significant difference between survivors and non-survivors.

## 9. Statistical Test Summary

The following table summarizes the six hypothesis tests conducted on the Titanic dataset.

In [20]:
summary = pd.DataFrame({
    "Hypothesis": [
        "Gender vs Survival",
        "Passenger Class vs Survival",
        "Age vs Survival",
        "Fare vs Survival",
        "Embarked Port vs Survival",
        "Family Size vs Survival"
    ],
    
    "Test": [
        "Chi-square",
        "Chi-square",
        "Welch's t-test",
        "Welch's t-test",
        "Chi-square",
        "Chi-square"
    ],
    
    "P-value": [
        p_gender,
        p_pclass,
        p_age,
        p_fare,
        p_embarked,
        p_family
    ],
    
    "Effect Size": [
        cramers_v_gender,
        cramers_v_pclass,
        "N/A",
        "N/A",
        cramers_v_embarked,
        cramers_v_family
    ],
    
    "Decision": [
        "Reject H0",
        "Reject H0",
        "Do not reject H0",
        "Reject H0",
        "Reject H0",
        "Reject H0"
    ]
})

summary

,Hypothesis,Test,P-value,Effect Size,Decision
0,Gender vs Survival,Chi-square,5.767311e-92,0.994831,Reject H0
1,Passenger Class vs Survival,Chi-square,3.519206e-02,0.126547,Reject H0
2,Age vs Survival,Welch's t-test,9.998233e-01,N/A,Do not reject H0
3,Fare vs Survival,Welch's t-test,6.914427e-04,N/A,Reject H0
4,Embarked Port vs Survival,Chi-square,3.039815e-02,0.129285,Reject H0
5,Family Size vs Survival,Chi-square,1.587440e-04,0.271015,Reject H0


### Overall Findings

The statistical analysis tested six relationships with passenger survival.

Five hypotheses showed statistically significant results at the 0.05 significance level. These relationships involved gender, passenger class, fare, embarkation port, and family size.

The age analysis did not show a statistically significant difference in mean age between survivors and non-survivors.

The effect size results show a strong association between gender and survival, while passenger class, embarkation port, and family size show weaker associations.

## Overall Findings

The statistical analysis tested six relationships with passenger survival.

Five hypotheses showed statistically significant results at the 0.05 significance level. These relationships involved gender, passenger class, fare, embarked port, and family size.

The analysis found a statistically significant association between gender and survival. Passenger class also showed a statistically significant association with survival.

Fare differed significantly between survivors and non-survivors. Embarked port and family size also showed statistically significant associations with survival.

Age did not show a statistically significant difference between survivors and non-survivors. Therefore, the null hypothesis for age was not rejected.

Overall, the statistical tests show that several passenger characteristics were associated with survival in the Titanic dataset.

## Statistical Conclusions

The hypothesis testing results provide statistical evidence about factors associated with passenger survival.

Gender showed the strongest relationship with survival, with a Cramer's V of approximately 0.995.

Passenger class showed a statistically significant relationship with survival, with a Cramer's V of approximately 0.127.

Fare showed a statistically significant difference between survivors and non-survivors. The average fare was approximately 49.75 for survivors and 27.56 for non-survivors.

Embarked port showed a statistically significant association with survival, with a Cramer's V of approximately 0.129.

Family size also showed a statistically significant association with survival, with a Cramer's V of approximately 0.271.

Age did not show a statistically significant difference between survivors and non-survivors. The average age was approximately 30.27 in both groups.

These results are based on the Titanic dataset used for this analysis and should be interpreted within the scope of the available variables and observations.

## Limitations

This analysis uses the available Titanic dataset containing 418 passenger records and 11 variables.

The statistical tests examine relationships between selected variables and survival. They do not establish causal relationships.

Some variables contain missing values, so the relevant statistical tests use available observations after removing missing values where required.

The results apply to the dataset used in this project and should not be generalized beyond the dataset without additional analysis.

## Final Conclusion

The statistical analysis identified several significant relationships with passenger survival.

Gender, passenger class, fare, embarked port, and family size showed statistically significant results at the 0.05 significance level.

Age did not show a statistically significant difference between survivors and non-survivors.

The analysis demonstrates how hypothesis testing can be used to validate patterns identified during exploratory data analysis and support data-driven conclusions.